## 00. load package

In [1]:

import intake
import cartopy.crs as ccrs
import cartopy.feature as cf
import cmocean
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import easygems.healpix as egh
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from typing import Tuple, List, Optional, Union, Dict
import scipy.signal as signal
from global_land_mask import globe
import math
import time
import os
import pickle
from scipy import fft
import cmaps  
from typing import Optional
import sys
import matplotlib.colors as mcolors
import importlib
sys.path.append("/home/m/m301257/wave_tools")
from wave_tools.utils import dataarray_to_equatorial_latlon_grid,get_region_healpix_,create_cmap_from_string,dataarray_healpix_to_equatorial_latlon
from  wave_tools.plotting import get_cckw_envelope_curve,setup_map_axes,set_axis_for_wave    
# 重新导入修复后的模块
import wave_tools.spectral
import wave_tools.plotting
importlib.reload(wave_tools.spectral)
importlib.reload(wave_tools.plotting)
from wave_tools.spectral import WKSpectralAnalysis, SpectralConfig
from wave_tools.plotting import set_axis_for_wave
from wave_tools import matsuno as mp
print("="*70)
print("✅ 模块重新加载完成")
print("="*70)
import mpi4py
import logging
import glob

✅ 模块重新加载完成


In [2]:
# 网格转换参数

def dataarray_to_equatorial_latlon_grid(
    dataarray: xr.DataArray, grid_type: str, grid_dict: Optional[dict]
) -> xr.DataArray:
    """转换数据到赤道经纬度网格"""
    if grid_type == "latlon":
        return dataarray
    elif grid_type == "healpix":
        if grid_dict is None:
            raise ValueError("No grid_dict provided for healpix conversion.")
        return dataarray_healpix_to_equatorial_latlon(dataarray, **grid_dict)
    else:
        raise ValueError("Grid type not found.")




In [3]:
pwd

'/work/mh1498/m301257'

In [4]:
cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
# 创建数据保存目录
DATA_DIR = "/work/mh1498/m301257/processed_data"
os.makedirs(DATA_DIR, exist_ok=True)
ds = (cat.ICON.C5.AMIP_CNTL.to_dask()).sel(time=slice("1980", "1993"))
ds 


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


<xarray.Dataset> Size: 6TB
Dimensions:             (time: 5114, cell: 786432, level_full: 26,
                         level_half: 26)
Coordinates:
  * time                (time) datetime64[ns] 41kB 1980-01-01 ... 1993-12-31
  * level_full          (level_full) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
  * level_half          (level_half) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
    healpix             int64 8B 1
Dimensions without coordinates: cell
Data variables: (12/47)
    clivi               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    cllvi               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hus2m               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hfls                (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hfss                (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    pr                  (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    ...                  ...
    phalf               (time, level_half, cell) float32 418GB dask.array<chunksize=(32, 4, 16384), meta=np.ndarray>
    cell_elevation      (cell) float64 6MB dask.array<chunksize=(262144,), meta=np.ndarray>
    cell_sea_land_mask  (cell) int32 3MB dask.array<chunksize=(262144,), meta=np.ndarray>
    zg                  (level_full, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
    zghalf              (level_half, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
    dzghalf             (level_full, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
Attributes:
    CDI:                       Climate Data Interface version 2.4.0 (https://...
    Conventions:               CF-1.6
    source:                    https://gitlab.dkrz.de/icon/icon-mpim.git@6684...
    institution:               Max Planck Institute for Meteorology/Deutscher...
    title:                     ICON simulation
    references:                see MPIM/DWD publications
    comment:                   Lukas Kluft (kluftluka) on nid006406 (Linux 5....
    cdo_bitrounding_numbits:   13
    CDO:                       Climate Data Operators version 2.4.0 (https://...
    cdo_openmp_thread_number:  4
    history:                   Wed Mar 13 20:19:59 2024: ncrename -d cells,ce...
    NCO:                       netCDF Operators version 5.0.1 (Homepage = htt...

In [5]:
def process_var_data(var_name, experiment_name, dataset_key, save_dir, grid_dict, target_lat, target_lon, 
                     has_level=True, level_slice=(30, None)):
    """
    处理变量数据（支持3D和2D），转换网格并插值后保存
    
    Parameters:
    -----------
    var_name : str
        变量名 ('wa', 'hus', 'ta', 'pr')
    experiment_name : str
        实验名称（用于显示）
    dataset_key : str
        在catalog中的数据集键名
    save_dir : str
        保存目录
    grid_dict : dict
        网格转换参数
    target_lat : array
        目标纬度
    target_lon : array
        目标经度
    has_level : bool
        是否为3D数据（有level维度）。True=3D，False=2D
    level_slice : tuple
        level切片范围，仅当has_level=True时有效
    """
    import time
    
    data_type = "3D (多层级)" if has_level else "2D (时间序列)"
    print("="*70)
    print(f"🔄 处理 {var_name.upper()} - {experiment_name} [{data_type}]")
    print("="*70)
    
    # 创建变量和实验的子目录
    if has_level:
        exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}_layers")
    else:
        exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}")
    os.makedirs(exp_save_dir, exist_ok=True)
    print(f"📁 保存路径: {exp_save_dir}")
    
    # 加载数据（只读取元数据）
    print(f"📖 读取数据元信息...")
    
    if has_level:
        # 3D数据：需要选择level范围
        if var_name == "ua":
            var_full = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(
                time=slice("1980", "1993"),
                level_half=slice(*level_slice)
            )
            levels = var_full.level_half.values
            n_levels = len(levels)
            print(f"   层级范围: {levels[0]:.1f} - {levels[-1]:.1f}")
        elif var_name == "va":
            var_full = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(
                time=slice("1980", "1993"),
                level_half=slice(*level_slice)
            )
            levels = var_full.level_half.values
            n_levels = len(levels)
            print(f"   层级范围: {levels[0]:.1f} - {levels[-1]:.1f}")
        else:
            var_full = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(
                time=slice("1980", "1993"),
                level_full=slice(*level_slice)
            )
            levels = var_full.level_full.values
            n_levels = len(levels)
        
        print(f"✅ 数据信息:")
        print(f"   变量: {var_name}")
        print(f"   时间范围: 1980-1993")
        print(f"   总层数: {n_levels}")
        print(f"   层级范围: {levels[0]:.1f} - {levels[-1]:.1f}")
        print(f"   时间步数: {len(var_full.time)}")
    else:
        # 2D数据：只有时间维度
        var_full = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(
            time=slice("1980", "1993")
        )
        
        print(f"✅ 数据信息:")
        print(f"   变量: {var_name}")
        print(f"   时间范围: 1980-1993")
        print(f"   时间步数: {len(var_full.time)}")
    
    print("="*70)
    
    # 逐层处理（3D）或整体处理（2D）
    total_start_time = time.time()
    
    if has_level:
        # ========== 3D数据：逐层处理 ==========
        for idx, level in enumerate(levels, 1):
            layer_start_time = time.time()
            
            # 构建保存路径
            save_path = os.path.join(exp_save_dir, f"{var_name}_lev_{int(level):03d}.nc")
            
            # 检查是否已处理
            if os.path.exists(save_path):
                print(f"✅ [{idx}/{n_levels}] Level {int(level):3d} - 已存在，跳过")
                continue
            
            print(f"🔄 [{idx}/{n_levels}] 处理 Level {int(level):3d}...")
            
            try:
                # 1. 选择单层数据
                if var_name in ["ua", "va"]:
                    var_layer = var_full.sel(level_half=level)
                else:
                    var_layer = var_full.sel(level_full=level)
                print(f"   ├─ 选择层级完成")
                
                # 2. 转换到经纬度网格
                var_lonlat = dataarray_to_equatorial_latlon_grid(var_layer, 'healpix', grid_dict)
                print(f"   ├─ 网格转换完成: {var_lonlat.shape}")
                
                # 3. 插值到2°x2°
                var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
                print(f"   ├─ 插值完成: {var_2deg.shape}")
                
                # 4. 转换为Dataset并设置变量名，然后保存
                ds_to_save = var_2deg.to_dataset(name=var_name)
                ds_to_save.to_netcdf(save_path)
                
                # 计算时间
                layer_time = time.time() - layer_start_time
                elapsed_time = time.time() - total_start_time
                avg_time_per_layer = elapsed_time / idx
                remaining_layers = n_levels - idx
                estimated_remaining = avg_time_per_layer * remaining_layers
                
                print(f"   ✅ 保存完成: {os.path.basename(save_path)}")
                print(f"   ⏱️  本层耗时: {layer_time:.1f}s | 平均: {avg_time_per_layer:.1f}s/层")
                print(f"   📊 预计剩余时间: {estimated_remaining/60:.1f} 分钟")
                print()
                
            except Exception as e:
                print(f"   ❌ 处理失败: {str(e)}")
                continue
    
    else:
        # ========== 2D数据：整体处理 ==========
        save_path = os.path.join(exp_save_dir, f"{var_name}_2deg_interp.nc")
        
        # 检查是否已处理
        if os.path.exists(save_path):
            print(f"✅ 数据已存在，跳过处理")
            print(f"   文件: {save_path}")
        else:
            print(f"🔄 开始处理2D数据...")
            
            try:
                # 1. 转换到经纬度网格
                print(f"   ├─ 网格转换中...")
                var_lonlat = dataarray_to_equatorial_latlon_grid(var_full, 'healpix', grid_dict)
                print(f"   ├─ 网格转换完成: {var_lonlat.shape}")
                
                # 2. 插值到2°x2°
                print(f"   ├─ 插值中...")
                var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
                print(f"   ├─ 插值完成: {var_2deg.shape}")
                
                # 3. 转换为Dataset并设置变量名，然后保存
                print(f"   ├─ 保存中...")
                ds_to_save = var_2deg.to_dataset(name=var_name)
                ds_to_save.to_netcdf(save_path)
                
                total_time = time.time() - total_start_time
                print(f"   ✅ 保存完成: {os.path.basename(save_path)}")
                print(f"   ⏱️  总耗时: {total_time/60:.1f} 分钟")
                
            except Exception as e:
                print(f"   ❌ 处理失败: {str(e)}")
    
    total_time = time.time() - total_start_time
    print("="*70)
    print(f"✅ {var_name.upper()} - {experiment_name} 处理完成!")
    print(f"   总耗时: {total_time/60:.1f} 分钟")
    print(f"   保存目录: {exp_save_dir}")
    print("="*70)
    print()


## 循环转换数据-气压层

In [6]:



# 网格参数
grid_dict = {"nside": 256, "nest": True, "minmax_lat": 16}
target_lat = np.arange(-14, 14.1, 2.0)
target_lon = np.arange(0, 360, 2.0)

# 要处理的变量列表（3D和2D）
variables_3d = [
    
    # "ua", "va"
    # "pfull"
    
    ]  # 3D变量：需要逐层处理
variables_2d = [
            # "hfls", "hfss", 
            #     "rsdt", "rsut", "rlut",   
            #     "rsds", "rsus", "rlds", "rlus",
                # "sfcwind",
                # "ts","tas"
                # "hus2m"
                # "ps"
                "zg"
                ]
          
if variables_2d:
    # 设置保存目录
    LAYER_DIR = os.path.join(DATA_DIR, "2d_layers")
    os.makedirs(LAYER_DIR, exist_ok=True)
else:
    LAYER_DIR = os.path.join(DATA_DIR, "3d_layers")
    os.makedirs(LAYER_DIR, exist_ok=True)   
print("="*70)
print("🚀 开始处理所有变量")
print("="*70)
# print(f"3D变量: {len(variables_3d)} ({', '.join(variables_3d)})")
print(f"2D变量: {len(variables_2d)} ({', '.join(variables_2d)})")
print(f"实验数量: 3 (CNTL, P4K, 4CO2)")
print(f"目标分辨率: 2° x 2°")
print(f"纬度范围: -14° to 14°")
print("="*70)
print()

# 图片保存路径
fig_save_path = os.path.join("/home/m/m301257/", "fig") 
os.makedirs(fig_save_path, exist_ok=True)

print("="*70)
print("📁 数据保存路径设置完成")
print("="*70)
print(f"3D数据目录: {LAYER_DIR}")
print(f"图片目录: {fig_save_path}")
print("="*70)
print()

# 定义实验配置
experiments = {
    "cntl":  ("CNTL",   "AMIP_CNTL"),
    "4k":    ("P4K",    "AMIP_P4K"),
    "4co2":  ("4CO2",   "AMIP_4CO2"),
}

# 处理所有变量和实验
all_start_time = time.time()

# # 处理3D变量
for var_name in variables_3d:
    print("\n" + "="*70)
    print(f"📊 开始处理3D变量: {var_name.upper()}")
    print("="*70 + "\n")
    
    for exp_key, (exp_name, dataset_key) in experiments.items():
        try:
            process_var_data(
                var_name=var_name,
                experiment_name=exp_name,
                dataset_key=dataset_key,
                save_dir=LAYER_DIR,
                grid_dict=grid_dict,
                target_lat=target_lat,
                target_lon=target_lon,
                has_level=True  # 3D数据
            )
        except Exception as e:
            print(f"❌ {var_name.upper()} - {exp_name} 处理失败: {str(e)}")
            print()
            continue

# 处理2D变量
for var_name in variables_2d:
    print("\n" + "="*70)
    print(f"📊 开始处理2D变量: {var_name.upper()}")
    print("="*70 + "\n")
    
    for exp_key, (exp_name, dataset_key) in experiments.items():
        try:
            process_var_data(
                var_name=var_name,
                experiment_name=exp_name,
                dataset_key=dataset_key,
                save_dir=LAYER_DIR,
                grid_dict=grid_dict,
                target_lat=target_lat,
                target_lon=target_lon,
                has_level=False  # 2D数据
            )
        except Exception as e:
            print(f"❌ {var_name.upper()} - {exp_name} 处理失败: {str(e)}")
            print()
            continue

all_total_time = time.time() - all_start_time
print("\n" + "="*70)
print("🎉 所有变量和实验处理完成!")
# print(f"3D变量数: {len(variables_3d)}")
# print(f"2D变量数: {len(variables_2d)}")
print(f"实验数: {len(experiments)}")
print(f"总耗时: {all_total_time/60:.1f} 分钟 ({all_total_time/3600:.2f} 小时)")
print("="*70)

🚀 开始处理所有变量
2D变量: 1 (zg)
实验数量: 3 (CNTL, P4K, 4CO2)
目标分辨率: 2° x 2°
纬度范围: -14° to 14°

📁 数据保存路径设置完成
3D数据目录: /work/mh1498/m301257/processed_data/2d_layers
图片目录: /home/m/m301257/fig


📊 开始处理2D变量: ZG

🔄 处理 ZG - CNTL [2D (时间序列)]
📁 保存路径: /work/mh1498/m301257/processed_data/2d_layers/zg_cntl
📖 读取数据元信息...
❌ ZG - CNTL 处理失败: "'time' is not a valid dimension or coordinate for Dataset with dimensions FrozenMappingWarningOnValuesAccess({'level_full': 26, 'cell': 786432})"

🔄 处理 ZG - P4K [2D (时间序列)]
📁 保存路径: /work/mh1498/m301257/processed_data/2d_layers/zg_p4k
📖 读取数据元信息...


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


❌ ZG - P4K 处理失败: "'time' is not a valid dimension or coordinate for Dataset with dimensions FrozenMappingWarningOnValuesAccess({'level_full': 26, 'cell': 786432})"

🔄 处理 ZG - 4CO2 [2D (时间序列)]
📁 保存路径: /work/mh1498/m301257/processed_data/2d_layers/zg_4co2
📖 读取数据元信息...
❌ ZG - 4CO2 处理失败: "'time' is not a valid dimension or coordinate for Dataset with dimensions FrozenMappingWarningOnValuesAccess({'level_full': 26, 'cell': 786432})"


🎉 所有变量和实验处理完成!
实验数: 3
总耗时: 0.0 分钟 (0.00 小时)


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


## 合并变量，修改变量名称，合并数据

In [7]:
def change_variable_name_and_merge_per_folder(in_paths, base_out_path, pattern,
                                              old_var_name, new_var_name,
                                              merged_file_name="wa_all_levels.nc",
                                              skip_existing=True,
                                              has_level=True):
    """
    批量更改NetCDF文件中的变量名，添加level维度（如果需要），并在每个子文件夹生成合并文件
    同时保留输入子文件夹结构
    
    Parameters:
    -----------
    skip_existing : bool
        如果为True，跳过已存在的单个文件和合并文件（默认True）
    has_level : bool
        如果为True，处理3D数据（从文件名提取level维度）
        如果为False，处理2D数据（直接合并，不添加level维度）
    """
    import os
    import glob
    import xarray as xr

    for in_path in in_paths:
        # 当前输入文件夹名称
        folder_name = os.path.basename(os.path.normpath(in_path))
        out_path = os.path.join(base_out_path, folder_name)
        os.makedirs(out_path, exist_ok=True)
        
        # 检查合并文件是否已存在
        merged_file = os.path.join(out_path, merged_file_name)
        if skip_existing and os.path.exists(merged_file):
            print(f"✅ {folder_name}: 合并文件已存在，跳过处理")
            print(f"   文件: {merged_file}")
            continue

        file_pattern = os.path.join(in_path, pattern)
        input_files = sorted(glob.glob(file_pattern))
        
        if not input_files:
            print(f"⚠️ 文件夹 {folder_name} 没有匹配的文件 (pattern: {pattern})")
            continue
        
        data_type = "3D (多层级)" if has_level else "2D (单层/时间序列)"
        print(f"\n{'='*70}")
        print(f"🔄 处理文件夹: {folder_name} [{data_type}]")
        print(f"   输入路径: {in_path}")
        print(f"   输出路径: {out_path}")
        print(f"   找到文件数: {len(input_files)}")
        print(f"{'='*70}")
        
        datasets = []
        processed_count = 0
        skipped_count = 0

        for file in input_files:
            filename = os.path.basename(file)
            new_file = os.path.join(out_path, filename)

            # 检查单个文件是否已存在
            if skip_existing and os.path.exists(new_file):
                skipped_count += 1
                # 仍需加载用于合并
                try:
                    with xr.open_dataset(new_file) as ds:
                        if has_level:
                            # 3D数据：需要level维度
                            level = int(filename.split("_")[2].replace(".nc", ""))
                            ds = ds.expand_dims({"level": [level]}) if "level" not in ds.dims else ds
                        datasets.append(ds)
                except Exception as e:
                    print(f"⚠️ 跳过的文件加载失败: {filename} - {str(e)}")
                continue

            # 处理新文件
            try:
                if has_level:
                    # 3D数据：从文件名提取 level (例如: hus_lev_031.nc -> 31)
                    try:
                        level = int(filename.split("_")[2].replace(".nc", ""))
                    except:
                        print(f"⚠️ 无法从文件名提取 level: {filename}, 跳过")
                        continue
                
                with xr.open_dataset(file) as ds:
                    # 修改变量名（如果需要）
                    if old_var_name in ds and old_var_name != new_var_name:
                        ds = ds.rename({old_var_name: new_var_name})
                    
                    # 3D数据需要添加 level 维度
                    if has_level:
                        ds = ds.expand_dims({"level": [level]})
                    
                    # 保存到新路径
                    ds.to_netcdf(new_file, mode="w")
                    processed_count += 1
                    
                    if processed_count % 5 == 0 or processed_count == len(input_files):
                        print(f"✅ [{processed_count}/{len(input_files)}] 已处理: {filename}")
                    
                    datasets.append(ds)
            except Exception as e:
                print(f"❌ 处理失败: {filename} - {str(e)}")
                continue

        # 统计信息
        print(f"\n📊 处理统计:")
        print(f"   新处理: {processed_count} 个文件")
        print(f"   已跳过: {skipped_count} 个文件")
        print(f"   总计: {len(datasets)} 个文件用于合并")

        # 每个子文件夹单独合并
        if datasets:
            try:
                if has_level:
                    # 3D数据：沿level维度合并
                    ds_all = xr.concat(datasets, dim="level")
                else:
                    # 2D数据：直接合并（沿时间或其他维度）
                    ds_all = xr.concat(datasets, dim="time") if "time" in datasets[0].dims else xr.merge(datasets)
                
                var_data = ds_all[new_var_name]
                var_data.to_netcdf(merged_file)
                print(f"🎉 {folder_name} 合并完成!")
                print(f"   保存到: {merged_file}")
                print(f"   形状: {var_data.shape}")
                print(f"   维度: {list(var_data.dims)}")
            except Exception as e:
                print(f"❌ 合并失败: {str(e)}")
        else:
            print(f"⚠️ 文件夹 {folder_name} 没有可处理的文件！")


## merge_files

In [8]:
# input_folders1 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_p4k_layers",

# ]

# input_folders2 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/va_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/va_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/va_p4k_layers"
# ]

# input_folders3 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_p4k_layers"
# ]
# base_output_folder = "/work/mh1498/m301257"

# # 处理3D数据 (hus - 比湿)
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders1,
#     base_out_path=base_output_folder,
#     pattern="ua_lev_*.nc",
#     old_var_name="ua",
#     new_var_name="ua",
#     has_level=True,  # 3D数据，有level维度,
#     merged_file_name="ua_all_levels.nc"
# )

# # 处理3D数据 (ta - 温度)
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders2,
#     base_out_path=base_output_folder,
#     pattern="va_lev_*.nc",
#     old_var_name="va",
#     new_var_name="va",
#     has_level=True , # 3D数据，有level维度
#     merged_file_name="va_all_levels.nc"
# )


# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders3,
#     base_out_path=base_output_folder,
#     pattern="pfull_lev_*.nc",
#     old_var_name="pfull",
#     new_var_name="pfull",
#     has_level=True , # 3D数据，有level维度
#     merged_file_name="pfull_all_levels.nc"
# )